# 📦 Loading Prepared Dataset Bundle

This notebook fetches a pre-built and validated **τ-Knowledge banking_knowledge** training data bundle,
verifies its integrity, and explores its contents.

## Data-First Principle

- Learner notebooks **do not run SDG (Synthetic Data Generation)**.
- Operators/advanced users pre-generate bundles for download and use.
- If the bundle is invalid, training does not start; you are directed to the authoring path.

## Bundle Structure

```
tau-knowledge-v1/
├── manifest.json          # Bundle metadata
├── checksums.sha256       # Integrity checksums
├── canonical/             # Canonical training/validation data
├── training/lora/         # LoRA backend-specific format
├── training/osft/         # OSFT backend-specific format
├── kb/                    # Knowledge base snapshot
├── metadata/              # Provenance and split metadata
└── reports/               # Quality reports
```

In [ ]:
"""Fetch the prepared dataset bundle."""

import os
import subprocess
import sys
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_bundle_config, PROJECT_ROOT

load_env()

release_config = load_bundle_config()
bundle_cfg = release_config.get("bundle", {})
bundle_name = f"{bundle_cfg.get('name', 'tau-knowledge')}-{bundle_cfg.get('version', 'v1')}"
default_bundle_path = Path(bundle_cfg.get("base_path", f"data/prepared/{bundle_name}"))

if not default_bundle_path.is_absolute():
    default_bundle_path = PROJECT_ROOT / default_bundle_path

print(f"Bundle name: {bundle_name}")
print(f"Default path: {default_bundle_path}")

# Try fetching via the fetch script
fetch_script = PROJECT_ROOT / "scripts" / "fetch_prepared_dataset.py"

if default_bundle_path.exists() and (default_bundle_path / "manifest.json").exists():
    print(f"\n✅ Bundle already exists: {default_bundle_path}")
    bundle_path = default_bundle_path
elif fetch_script.exists():
    print(f"\n📥 Downloading bundle...")
    result = subprocess.run(
        [sys.executable, str(fetch_script), "--release", "configs/data-release.yaml"],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        print("✅ Bundle download complete")
        bundle_path = default_bundle_path
    else:
        print(f"❌ Download failed:\n{result.stderr}")
        raise RuntimeError("Cannot fetch bundle. Check network and S3/PVC settings.")
else:
    # Try the data module's fetch_bundle
    from rhoai_model_training_lab.data import fetch_bundle
    bundle_path = fetch_bundle(release_config)
    print(f"✅ Bundle loaded: {bundle_path}")

In [ ]:
"""Validate checksums and compatibility."""

from rhoai_model_training_lab.data import BundleManager, validate_prepared_dataset

# Comprehensive validation
print("📋 Validating bundle integrity...\n")
validation = validate_prepared_dataset(bundle_path)

if validation["valid"]:
    print("✅ Bundle validation passed")
else:
    print("❌ Bundle validation failed")
    for err in validation["errors"]:
        print(f"  ❌ {err}")

if validation["warnings"]:
    print("\n⚠️  Warnings:")
    for warn in validation["warnings"]:
        print(f"  ⚠️  {warn}")

print("\nSummary:")
for k, v in validation.get("summary", {}).items():
    print(f"  {k}: {v}")

if not validation["valid"]:
    raise RuntimeError(
        "Bundle validation failed — cannot proceed with training.\n"
        "Regenerate the bundle through the authoring path (data_preparation/)."
    )

In [ ]:
"""Load and display the bundle manifest."""

import json
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Display manifest
table = Table(title="📄 Bundle Manifest", show_header=True)
table.add_column("Item", style="bold cyan")
table.add_column("Value")

manifest_items = [
    ("Bundle Name", manifest.bundle_name),
    ("Version", manifest.bundle_version),
    ("Created", manifest.created_at),
    ("τ Version", manifest.tau_version),
    ("τ Commit SHA", manifest.tau_commit_sha[:12] + "..." if manifest.tau_commit_sha else "N/A"),
    ("Model ID", manifest.model_id),
    ("Model Revision", manifest.model_revision),
    ("Tokenizer ID", manifest.tokenizer_id),
    ("Training Samples", str(manifest.canonical_train_count)),
    ("Validation Samples", str(manifest.canonical_validation_count)),
    ("Split Policy", manifest.split_policy),
]

for label, value in manifest_items:
    table.add_row(label, value)

console.print(table)

# Type distribution
if manifest.sample_type_distribution:
    print("\n📊 Sample type distribution:")
    for stype, count in sorted(manifest.sample_type_distribution.items()):
        total = manifest.canonical_train_count + manifest.canonical_validation_count
        pct = (count / total * 100) if total > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"  {stype:<25} {count:>5} ({pct:5.1f}%) {bar}")

In [ ]:
"""Show sample statistics and type distribution."""

from collections import Counter

# Load canonical samples
train_samples = mgr.get_samples("train")
val_samples = mgr.get_samples("validation")

print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Total samples: {len(train_samples) + len(val_samples)}")

# Type distribution breakdown
print("\n--- Training data type statistics ---")
train_types = Counter(s.sample_type for s in train_samples)
for stype, count in train_types.most_common():
    pct = count / len(train_samples) * 100
    print(f"  {stype.value:<25} {count:>5} ({pct:.1f}%)")

print("\n--- Validation data type statistics ---")
val_types = Counter(s.sample_type for s in val_samples)
for stype, count in val_types.most_common():
    pct = count / len(val_samples) * 100
    print(f"  {stype.value:<25} {count:>5} ({pct:.1f}%)")

# Message length statistics
print("\n--- Message length statistics ---")
msg_counts = [len(s.messages) for s in train_samples]
print(f"  Average messages: {sum(msg_counts)/len(msg_counts):.1f}")
print(f"  Min/Max: {min(msg_counts)} / {max(msg_counts)}")

# Tool usage stats
samples_with_tools = sum(1 for s in train_samples if s.tools)
print(f"\n  Samples with tool schemas: {samples_with_tools} ({samples_with_tools/len(train_samples)*100:.1f}%)")

# Validation status
status_counts = Counter(s.validation_status for s in train_samples)
print("\n--- Validation status ---")
for status, count in status_counts.most_common():
    print(f"  {status.value:<20} {count:>5}")

In [ ]:
"""Preview example samples (one per type)."""

from rich.syntax import Syntax

print("=" * 70)
print("📝 Sample preview by type (one per type from training data)")
print("=" * 70)

seen_types = set()
for sample in train_samples:
    if sample.sample_type in seen_types:
        continue
    seen_types.add(sample.sample_type)

    print(f"\n{'─' * 70}")
    print(f"Type: {sample.sample_type.value}")
    print(f"ID: {sample.sample_id}")
    if sample.source_doc_ids:
        print(f"Source documents: {', '.join(sample.source_doc_ids[:3])}")
    if sample.scenario_family:
        print(f"Scenario family: {sample.scenario_family}")
    print(f"Messages: {len(sample.messages)}")
    if sample.tools:
        tool_names = [t.function.name for t in sample.tools]
        print(f"Tools: {', '.join(tool_names)}")
    print()

    # Show messages (truncated)
    for msg in sample.messages:
        role_icon = {"system": "🔧", "user": "👤", "assistant": "🤖", "tool": "🔨"}.get(msg.role, "❓")
        content = msg.content or ""
        if len(content) > 300:
            content = content[:300] + "..."
        print(f"  {role_icon} [{msg.role}]: {content}")
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"     🔧 tool_call: {tc.function.name}({tc.function.arguments[:100]})")

print(f"\n{'=' * 70}")
print(f"Previewed samples from {len(seen_types)} types.")

In [ ]:
"""Print validation summary."""

table = Table(title="✅ Dataset Bundle Validation Summary", show_header=True)
table.add_column("Check Item", style="bold")
table.add_column("Result")

summary_items = [
    ("Bundle path", str(bundle_path)),
    ("Manifest valid", "✅"),
    ("Checksum verification", "✅" if validation.get("summary", {}).get("checksums_ok") else "❌"),
    ("Training samples", f"{len(train_samples):,}"),
    ("Validation samples", f"{len(val_samples):,}"),
    ("Type count", str(len(seen_types))),
    ("Samples with tools", f"{samples_with_tools:,}"),
    ("Target model", manifest.model_id),
    ("Bundle status", "✅ Ready for training" if validation["valid"] else "❌ Validation failed"),
]

for label, value in summary_items:
    table.add_row(label, value)

console.print(table)

print("\nNext steps:")
print("  📓 03_lora_finetuning.ipynb  — Run LoRA fine-tuning")
print("  📓 04_osft_finetuning.ipynb  — Run OSFT fine-tuning")
print("  Both training methods proceed independently from the same base model and data.")